In [1]:
#necessary libraries
!pip install xgboost
import pandas as pd
import numpy as np
import seaborn as sns
import os
from sklearn.model_selection import train_test_split, cross_val_score, GridSearchCV, KFold
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import mean_squared_error, r2_score
from xgboost import XGBRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.metrics import make_scorer

In [2]:
#loading training and test data
appartments = pd.read_csv("appartments_train.csv")
test_df = pd.read_csv("appartments_test.csv")

In [3]:
#Missing values

#year built missing col
appartments['year_built_missing'] = appartments['year_built'].isna().astype(int)
test_df['year_built_missing'] = test_df['year_built'].isna().astype(int)
appartments['year_built_missing'] = appartments['year_built'].isna().map({True: 'yes', False: 'no'})
test_df['year_built_missing'] = test_df['year_built'].isna().map({True: 'yes', False: 'no'})
median_year = appartments['year_built'].median()
appartments['year_built'] = appartments['year_built'].fillna(median_year)
test_df['year_built'] = test_df['year_built'].fillna(median_year)

# columns which starts with 'dist_'
dist_cols = [col for col in appartments.columns if col.startswith('dist_')]
# imputation: using median from train (appartments) for both datasets
for col in dist_cols:
    median_val = appartments[col].median()
    appartments[col] = appartments[col].fillna(median_val)
    test_df[col] = test_df[col].fillna(median_val)
    
appartments['obj_type'] = appartments['obj_type'].fillna('missing')
appartments['build_mat'] = appartments['build_mat'].fillna('missing')
test_df['obj_type'] = test_df['obj_type'].fillna('missing')
test_df['build_mat'] = test_df['build_mat'].fillna('missing')

most_common_floor_no = appartments['floor_no'].mode()[0]
def impute_floor_no(row, most_common):
    if pd.isna(row['floor_no']):
        if not pd.isna(row['floor_max']):
            return max(round(row['floor_max'] / 2), 1.0)
        elif row['has_lift'] == 'yes':
            return 5.0
        else:
            return most_common
    else:
        return row['floor_no']

appartments['floor_no'] = appartments.apply(lambda row: impute_floor_no(row, most_common_floor_no), axis=1)
# Imputation in a test with the same logic and the same modal value from train
test_df['floor_no'] = test_df.apply(lambda row: impute_floor_no(row, most_common_floor_no), axis=1)


floor_max_median_train = appartments['floor_max'].median()
#  floor_max based on floor_no
def impute_floor_max(row, median_val):
    if pd.isna(row['floor_max']):
        if not pd.isna(row['floor_no']):
            if 1 <= row['floor_no'] <= 4:
                return 4
            elif 5 <= row['floor_no'] <= 10:
                return 10
            elif row['floor_no'] > 10:
                return row['floor_no']
        else:
            return median_val  # fallback if floor_no is missing too
    return row['floor_max']
appartments['floor_max'] = appartments.apply(lambda row: impute_floor_max(row, floor_max_median_train), axis=1)
test_df['floor_max'] = test_df.apply(lambda row: impute_floor_max(row, floor_max_median_train), axis=1)

# has_lift based on floor_max
def infer_lift(row):
    if pd.isna(row['has_lift']):
        if not pd.isna(row['floor_max']) and row['floor_max'] > 4:
            return 'yes'
        else:
            return 'no'
    return row['has_lift']

appartments['has_lift'] = appartments.apply(infer_lift, axis=1)
test_df['has_lift'] = test_df.apply(infer_lift, axis=1)

#dropign cond_class
appartments.drop(columns=['cond_class'], inplace=True)
test_df.drop(columns=['cond_class'], inplace=True)

filtered_df = appartments[['year_built', 'infrastructure_quality']].dropna()
mean_quality_by_year = (
    filtered_df
    .groupby('year_built')['infrastructure_quality']
    .mean()
)
appartments['mean_quality_for_year'] = appartments['year_built'].map(mean_quality_by_year)
appartments['infrastructure_quality'] = appartments['infrastructure_quality'].fillna(
    appartments['mean_quality_for_year']
)    
    

test_df['mean_quality_for_year'] = test_df['year_built'].map(mean_quality_by_year)
test_df['infrastructure_quality'] = test_df['infrastructure_quality'].fillna(
    test_df['mean_quality_for_year']
)
    
appartments.drop(columns='mean_quality_for_year', inplace=True)    
test_df.drop(columns='mean_quality_for_year', inplace=True)


## Convert src_month to datetime and create variables: year, month number, quarter
for df in [appartments, test_df]:
    df['src_month'] = pd.to_datetime(df['src_month'], format="%Y-%m", errors='coerce')
    df['src_year'] = df['src_month'].dt.year
    df['src_month_num'] = df['src_month'].dt.month
    df['src_quarter_str'] = df['src_month_num'].apply(
        lambda m: '1st quarter' if m in [1, 2, 3]
        else '2nd quarter' if m in [4, 5, 6]
        else '3rd quarter' if m in [7, 8, 9]
        else '4th quarter'
    )


src_quarter_str
2nd quarter    48838
4th quarter    39708
1st quarter    39268
3rd quarter    28640
Name: count, dtype: int64
src_year
2024    88106
2023    68348
Name: count, dtype: int64


In [4]:
#log transformation

# Columns to log-transform (right-skewed)
log_cols = ['dim_m2', 'floor_no', 'floor_max',  'n_poi', 'dist_sch', 'dist_post', 'dist_kind', 'dist_centre',  'dist_uni', 
    'dist_pharma','dist_clinic', 'dist_rest', 'estimated_maintenance_cost']   

created_log_vars_app = []
for col in log_cols:
    if col in appartments.columns:
        log_col = 'log_' + col
        appartments[log_col] = np.log1p(appartments[col].clip(lower=0))
        created_log_vars_app.append(log_col)
        
appartments['log_price_z'] = np.log1p(appartments['price_z'])


created_log_vars_test = []
for col in log_cols:
    if col in test_df.columns:
        log_col = 'log_' + col
        test_df[log_col] = np.log1p(test_df[col].clip(lower=0))
        created_log_vars_test.append(log_col)

In [5]:
numerical_cols1 = ['n_rooms', 'year_built','market_volatility',  
                 'src_year','neighborhood_crime_rate','popularity_index', 'global_economic_index',
                 'log_dim_m2', 'log_floor_no', 'log_floor_max', 'log_n_poi', 'log_dist_sch', 'log_dist_post', 'log_dist_kind', 
                  'log_dist_centre', 'log_dist_uni', 'log_dist_pharma', 'log_dist_clinic', 'log_dist_rest', 
                   'log_estimated_maintenance_cost']

categorical_cols = ['obj_type', 'own_type', 'build_mat', 'has_park','has_balcony', 'has_lift', 'has_sec', 'has_store',
                    'loc_code','year_built_missing', 'src_quarter_str']

drop_cols = ['unit_id','price_z', 'log_price_z', 'dim_m2', 'floor_no', 'floor_max',  'n_poi', 'dist_sch', 'green_space_ratio',
                              'dist_post','dist_kind', 'dist_centre',  'dist_uni', 'dist_pharma','dist_clinic', 'dist_rest',
                               'infrastructure_quality',  'estimated_maintenance_cost']

X = appartments.drop(columns=drop_cols)
y = appartments['log_price_z']

numerical_cols1   = [c for c in numerical_cols1   if c in X.columns]
categorical_cols  = [c for c in categorical_cols  if c in X.columns]

missing_cols = set(X.columns) - set(test_df.columns)
if missing_cols:
    print("Missing columns in test_df:", missing_cols)
else:
    print("All columns from X are present in test_df.")


    
num_tf = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scale',   StandardScaler())
])

cat_tf = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('oh',      OneHotEncoder(handle_unknown='ignore', sparse_output=True))
])

preprocessor = ColumnTransformer([
    ('num', num_tf, numerical_cols1),
    ('cat', cat_tf, categorical_cols)
])


All columns from X are present in test_df.


In [6]:
#final XGB
best_xgb = XGBRegressor(
    tree_method="hist",
    objective="reg:squarederror",
    eval_metric="rmse",
    n_estimators=600,
    max_depth=8,
    learning_rate=0.02,
    subsample=0.7,
    colsample_bytree=0.8,
    n_jobs=-1,
    random_state=42
)


final_pipe = make_pipeline(preprocessor, best_xgb)
final_pipe.fit(X, y)

missing_cols = set(X.columns) - set(test_df.columns)
for col in missing_cols:
    test_df[col] = np.nan 

X_test = test_df[X.columns]  
log_preds = final_pipe.predict(X_test)
price_preds = np.expm1(log_preds)


os.makedirs("final", exist_ok=True)
pred_df = test_df[['unit_id']].copy()
pred_df['price_z_pred'] = price_preds
pred_df.to_csv("final/final_xgb_predictions_Kosiec_Pełka.csv", index=False)

print("Predictions saved to final/final_xgb_predictions_Kosiec_Pełka.csv")


Predictions saved to final/final_xgb_predictions_Kosiec_Pełka.csv
